# Optimizer race on a ravine

Take $f(x, y) = \tfrac{1}{2}(x^2 + 100\,y^2)$, a ravine whose curvature in $y$ is 100 times its curvature
in $x$. Gradient descent on it diverges as soon as the learning rate passes $2/100 = 0.02$, so the
steep direction sets the largest step the flat direction is allowed to take. Here you write three
optimizers in NumPy, plain gradient descent, gradient descent with momentum, and Adam, run each for 100
steps from the same start, and see how far each one gets.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np


def loss(point):
    """f(x, y) = (x^2 + 100 y^2) / 2 at point = (x, y)."""
    x, y = point
    return 0.5 * (x ** 2 + 100.0 * y ** 2)


def gradient(point):
    """The gradient of f at point, as the array (x, 100 y)."""
    x, y = point
    return np.array([x, 100.0 * y])


START = np.array([-8.0, 2.0])


def show(name, losses):
    """Print the loss after steps 20, 40, 60, 80 and 100."""
    print(f"{name:9s}", " ".join(f"{losses[i]:10.3g}" for i in range(19, 100, 20)))

## Part 1 (coding task): Plain gradient descent

Write `run_sgd(eta=0.019, steps=100)`. Start at `START`, take `steps` steps of gradient descent with
learning rate `eta`, and return the list of the loss after each step, so the list has `steps` entries
and the last one is the loss at the final point. The default rate, 0.019, is just under the largest one
gradient descent survives on this ravine.

In [ ]:
def run_sgd(eta=0.019, steps=100):
    """Losses after each of `steps` gradient descent steps from START."""
    # TODO
    raise NotImplementedError

In [ ]:
losses = np.asarray(run_sgd(), dtype=float)
assert losses.shape == (100,), f"expected 100 losses, one per step, got shape {losses.shape}"
# Each step scales x by (1 - eta) and y by (1 - 100 eta), so the loss has a closed form.
t = np.arange(1, 101)
expected = 0.5 * (64 * 0.981 ** (2 * t) + 400 * 0.81 ** t)
assert np.isclose(losses[0], expected[0]), f"loss after step 1 is {losses[0]:.4f}, expected {expected[0]:.4f}: record the loss after the step, not before it"
assert np.allclose(losses, expected, rtol=1e-6), "the losses drift from the closed form: x should shrink by (1 - eta) and y by (1 - 100 eta) at every step"
assert np.isclose(losses[-1], 0.6902, atol=1e-3), f"final loss {losses[-1]:.4f}, expected 0.690"
over_the_limit = np.asarray(run_sgd(eta=0.021), dtype=float)
assert over_the_limit[-1] > 1e6, "at eta = 0.021 the y coordinate should blow up: every step multiplies it by -1.1"
show("sgd", losses)
print("Part 1 passed")

## Part 2 (coding task): Momentum

Write `run_momentum(eta=0.01, beta=0.9, steps=100)`. Keep a velocity `v`, starting at zero. At each
step set `v` to `beta` times itself plus the gradient at the current point, then move the point by
`eta` times `v` against it. Return the losses after each step, as in Part 1. The velocity remembers
past gradients: the back-and-forth gradients in $y$ cancel inside it, while the steady gradients in
$x$ pile up.

In [ ]:
def run_momentum(eta=0.01, beta=0.9, steps=100):
    """Losses after each of `steps` momentum steps from START."""
    # TODO
    raise NotImplementedError

In [ ]:
losses = np.asarray(run_momentum(), dtype=float)
assert losses.shape == (100,), f"expected 100 losses, got shape {losses.shape}"
assert np.isclose(losses[0], 31.3632, rtol=1e-4), f"loss after step 1 is {losses[0]:.4f}, expected 31.3632: with v starting at zero the first step is a plain gradient step of size eta"
assert np.isclose(losses[1], 192.177, rtol=1e-4), f"loss after step 2 is {losses[1]:.3f}, expected 192.177: the velocity keeps 0.9 of itself and adds the whole new gradient"
assert np.isclose(losses[-1], 5.865e-4, rtol=1e-2), f"final loss {losses[-1]:.3e}, expected 5.86e-4"
no_memory = np.asarray(run_momentum(eta=0.019, beta=0.0), dtype=float)
assert np.allclose(no_memory, run_sgd(eta=0.019)), "with beta = 0 momentum should be plain gradient descent at the same learning rate"
show("momentum", losses)
print("Part 2 passed")

## Part 3 (coding task): Adam

Write `run_adam(eta=0.3, beta1=0.9, beta2=0.999, eps=1e-8, steps=100)`. Keep two running averages,
`m` of the gradient and `v` of its square, both starting at zero. At step $t = 1, 2, \dots$ update them as

$$
m \leftarrow \beta_1 m + (1 - \beta_1)\, g, \qquad v \leftarrow \beta_2 v + (1 - \beta_2)\, g^2,
$$

correct them to $\hat{m} = m / (1 - \beta_1^t)$ and $\hat{v} = v / (1 - \beta_2^t)$, and move the
point by `eta` times $\hat{m} / (\sqrt{\hat{v}} + \texttt{eps})$ against the gradient, entry by
entry. Return the losses after each step. Adam gets a learning rate fifteen times the largest one
gradient descent survives, and the division by each coordinate's own gradient size is what lets it.

In [ ]:
def run_adam(eta=0.3, beta1=0.9, beta2=0.999, eps=1e-8, steps=100):
    """Losses after each of `steps` Adam steps from START."""
    # TODO
    raise NotImplementedError

In [ ]:
losses = np.asarray(run_adam(), dtype=float)
assert losses.shape == (100,), f"expected 100 losses, got shape {losses.shape}"
assert np.isclose(losses[0], 174.145, rtol=1e-4), f"loss after step 1 is {losses[0]:.3f}, expected 174.145: after bias correction the first step moves each coordinate by exactly eta against the sign of its gradient, to (-7.7, 1.7)"
assert np.isclose(losses[19], 6.8296, rtol=1e-3), f"loss after step 20 is {losses[19]:.4f}, expected 6.830"
assert np.isclose(losses[-1], 2.101e-4, rtol=1e-2), f"final loss {losses[-1]:.3e}, expected 2.10e-4"
show("adam", losses)
print("Part 3 passed")

## Solution

### Part 1: Plain gradient descent

In [ ]:
def run_sgd(eta=0.019, steps=100):
    """Losses after each of `steps` gradient descent steps from START."""
    point = START.copy()
    losses = []
    for _ in range(steps):
        point = point - eta * gradient(point)
        losses.append(loss(point))
    return losses

**Why it works.** Each coordinate moves on its own: a step multiplies $x$ by $1 - \eta$ and $y$ by
$1 - 100\eta$. At $\eta = 0.019$ the $y$ factor is $-0.9$, so $y$ flips sign every step and shrinks
slowly, while the $x$ factor is $0.981$ and $x$ is still $0.981^{100} \approx 0.15$ of its start after
100 steps. The steep direction caps the learning rate, and the flat direction crawls at that cap. Push
$\eta$ to 0.021 and the $y$ factor is $-1.1$: the loss explodes along $y$ alone.

### Part 2: Momentum

In [ ]:
def run_momentum(eta=0.01, beta=0.9, steps=100):
    """Losses after each of `steps` momentum steps from START."""
    point = START.copy()
    v = np.zeros(2)
    losses = []
    for _ in range(steps):
        v = beta * v + gradient(point)
        point = point - eta * v
        losses.append(loss(point))
    return losses

**Why it works.** With $v$ starting at zero the first step is a plain gradient step, and after that
the velocity is a decaying sum of past gradients. The gradients in $y$ alternate in sign and mostly
cancel inside $v$, while the gradients in $x$ all point the same way and add up to about
$1 / (1 - \beta) = 10$ times a single one. Momentum ends about a thousand times lower than gradient
descent on roughly half the learning rate. It also survives larger rates: on this ravine it only
diverges past $\eta \approx 0.038$, which is $2(1 + \beta) / 100$, about twice the limit for plain
gradient descent.

### Part 3: Adam

In [ ]:
def run_adam(eta=0.3, beta1=0.9, beta2=0.999, eps=1e-8, steps=100):
    """Losses after each of `steps` Adam steps from START."""
    point = START.copy()
    m = np.zeros(2)
    v = np.zeros(2)
    losses = []
    for t in range(1, steps + 1):
        g = gradient(point)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g ** 2
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)
        point = point - eta * m_hat / (np.sqrt(v_hat) + eps)
        losses.append(loss(point))
    return losses

**Why it works.** Adam divides each coordinate's step by a running estimate of that coordinate's
gradient size, so the steep $y$ direction and the flat $x$ direction both move by about $\eta$ per
step at first. That is why it can take $\eta = 0.3$, fifteen times what gradient descent survives,
and why its very first step lands on $(-7.7, 1.7)$ exactly. The bias correction matters here: without
it the first step would be $\sqrt{10} \approx 3.2$ times longer. Near the minimum the steps shrink as
the averaged gradients fall. Adam sits between momentum and plain gradient descent at step 60 and is
ahead of both by step 100.